In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

## Generate Similarity Matrix

USM contains the following 
### One Hot Categoricals
1. Airing Status
2. Genres
3. Themes
4. Demographics
5. Rating
6. Source

### Continuous Numerics
1. Score
2. Members
3. Favorites
4. Related_Entries
5. Year
6. Episodes

Missing values are present in Score and Episodes.

In [2]:
USM = pd.read_csv('USM.csv', index_col=0)
USM = USM.loc[~USM.index.duplicated(keep='first')] # Duplicates present from shows continued airing between multiple seasons but still classified as one.
USM.describe()

,Is_Finished,Genre_Comedy,Genre_Romance,Genre_Supernatural,Genre_Drama,Genre_Ecchi,Genre_Action,Genre_Fantasy,Genre_Sci-Fi,Genre_Horror,...,Source_4-koma manga,Source_Light novel,Source_Manga,Source_Web manga,Score,Members,Favorites,Related_Entries,Year,Episodes
count,11241.000000,11241.000000,11241.000000,11241.000000,11241.000000,11241.000000,11241.000000,11241.000000,11241.000000,11241.000000,...,11241.00000,11241.000000,11241.000000,11241.000000,8366.000000,1.124100e+04,11241.000000,11241.000000,11241.000000,11049.000000
mean,0.976426,0.359132,0.087804,0.073481,0.098835,0.039320,0.256650,0.257184,0.115203,0.020105,...,0.02224,0.069745,0.238057,0.043679,6.653940,7.416988e+04,809.870919,1.244551,2017.179699,12.585664
std,0.151726,0.479767,0.283022,0.260936,0.298453,0.194365,0.436804,0.437101,0.319281,0.140366,...,0.14747,0.254728,0.425913,0.204390,0.864496,2.331803e+05,6242.906730,1.256964,4.119036,35.498046
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,2.400000,3.500000e+01,0.000000,0.000000,2010.000000,1.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,6.060000,4.770000e+02,0.000000,0.000000,2014.000000,1.000000
50%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,6.640000,4.679000e+03,7.000000,2.000000,2017.000000,6.000000
75%,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,7.270000,3.766000e+04,84.000000,2.000000,2021.000000,13.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.00000,1.000000,1.000000,1.000000,9.310000,4.119103e+06,234719.000000,5.000000,2024.000000,1818.000000


Add scaling to avoid numerics from overpowering booleans.
We will be imputing with 0.

In [3]:
scaler = MinMaxScaler()

cols = ['Score', 'Members', 'Favorites', 'Related_Entries', 'Year', 'Episodes']
USM[cols] = scaler.fit_transform(USM[cols])

USM = USM.fillna(0)
USM

,Title,Is_Finished,Genre_Comedy,Genre_Romance,Genre_Supernatural,Genre_Drama,Genre_Ecchi,Genre_Action,Genre_Fantasy,Genre_Sci-Fi,...,Source_4-koma manga,Source_Light novel,Source_Manga,Source_Web manga,Score,Members,Favorites,Related_Entries,Year,Episodes
MAL_id,,,,,,,,,,,,,,,,,,,,,
8769,Ore no Imouto ga Konnani Kawaii Wake ga Nai,1,1,0,0,0,0,0,0,0,...,0,1,0,0,0.655572,0.170526,0.019236,0.8,0.0,0.006054
8525,Kami nomi zo Shiru Sekai,1,1,1,1,0,0,0,0,0,...,0,0,1,0,0.758321,0.156852,0.024783,0.8,0.0,0.006054
7674,Bakuman.,1,1,1,0,1,0,0,0,0,...,0,0,1,0,0.835022,0.154904,0.040606,0.4,0.0,0.013209
8861,Yosuga no Sora,1,0,1,0,1,1,0,0,0,...,0,0,0,0,0.523878,0.127091,0.011192,0.4,0.0,0.006054
8937,Toaru Majutsu no Index II,1,0,0,0,0,0,1,1,1,...,0,1,0,0,0.739508,0.114935,0.007375,0.8,0.0,0.012658
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58606,Girls & Panzer: Saishuushou Part 4 Specials,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0.707670,0.000709,0.000009,0.0,1.0,0.000550
58006,Science SARU x MBS Original Short Anime Daisak...,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0.529667,0.000122,0.000000,0.0,1.0,0.001651
58229,Harutsugeuo to Fuuraibou,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0.000000,0.000048,0.000000,0.0,1.0,0.000000


Display the similarity when compared against another entry

In [4]:
df_Similarity = pd.DataFrame(cosine_similarity(USM.drop(columns=['Title'])), 
                             index=USM.index,
                             columns=USM.index)

df_Similarity

MAL_id,8769,8525,7674,8861,8937,8795,9181,8407,8129,8424,...,58592,57834,55806,58226,58015,58606,58006,58229,58605,58308
MAL_id,,,,,,,,,,,,,,,,,,,,,
8769,1.000000,0.406365,0.407137,0.195852,0.319233,0.228320,0.255807,0.309611,0.281468,0.402346,...,0.260161,0.182138,0.256930,0.280386,0.258863,0.289219,0.241038,0.085784,0.105063,0.081381
8525,0.406365,1.000000,0.617261,0.343919,0.260587,0.231875,0.630239,0.613043,0.342502,0.474623,...,0.339732,0.337983,0.259976,0.283161,0.414350,0.293919,0.245483,0.085386,0.104575,0.081004
7674,0.407137,0.617261,1.000000,0.427283,0.255989,0.236715,0.448608,0.499756,0.353048,0.343856,...,0.359781,0.357692,0.195839,0.299868,0.437756,0.312678,0.261554,0.178924,0.109568,0.084871
8861,0.195852,0.343919,0.427283,1.000000,0.186016,0.234022,0.458559,0.379590,0.165454,0.347014,...,0.111626,0.109941,0.276925,0.116333,0.110267,0.126039,0.144466,0.094468,0.115698,0.089620
8937,0.319233,0.260587,0.255989,0.186016,1.000000,0.439843,0.301079,0.293307,0.213154,0.315058,...,0.247503,0.245756,0.315733,0.186432,0.246099,0.119171,0.133902,0.080374,0.196873,0.152497
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58606,0.289219,0.293919,0.312678,0.126039,0.119171,0.190671,0.274086,0.267441,0.261060,0.393307,...,0.427162,0.328859,0.616230,0.569750,0.425593,1.000000,0.436891,0.216286,0.264895,0.307781
58006,0.241038,0.245483,0.261554,0.144466,0.133902,0.129748,0.132282,0.223224,0.218340,0.254559,...,0.512164,0.510627,0.390261,0.558349,0.510942,0.436891,1.000000,0.399026,0.488705,0.252366
58229,0.085784,0.085386,0.178924,0.094468,0.080374,0.076948,0.080511,0.078193,0.074823,0.090360,...,0.394995,0.395911,0.198383,0.329311,0.395732,0.216286,0.399026,1.000000,0.544331,0.421637


## Test on Jujutsu Kaisen 
Show similarity features that JJK needs

In [5]:
i = USM.query('Title == "Jujutsu Kaisen"').index[0]
i = df_Similarity.loc[i].sort_values(ascending=False)[:11]

df_Results = USM.loc[i.keys()].copy()
df_Results.insert(0, 'Similarity', i)

colMask = df_Results.iloc[0].apply(lambda x : x != 0)
df_Results.loc[:, colMask]

,Similarity,Title,Is_Finished,Genre_Supernatural,Genre_Action,Genre_Award Winning,Demo_Shounen,Theme_School,Studio_MAPPA,Producer_Shueisha,...,Licensor_VIZ Media,Type_TV,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Members,Favorites,Related_Entries,Year,Episodes
MAL_id,,,,,,,,,,,,,,,,,,,,,
40748,1.000000,Jujutsu Kaisen,1,1,1,1,1,1,1,1,...,1,1,1,1,0.892909,0.665196,0.396487,0.8,0.714286,0.012658
51009,0.909200,Jujutsu Kaisen 2nd Season,1,1,1,0,1,1,1,1,...,0,1,1,1,0.920405,0.278167,0.102365,0.8,0.928571,0.012108
48561,0.820457,Jujutsu Kaisen 0 Movie,1,1,1,0,1,1,1,1,...,0,0,1,1,0.869754,0.264022,0.045454,0.8,0.785714,0.000000
56243,0.710336,Jujutsu Kaisen 2nd Season Recaps,1,1,1,0,1,1,1,0,...,0,0,1,1,0.761216,0.006859,0.000831,0.0,0.928571,0.000550
38000,0.658897,Kimetsu no Yaiba,1,1,1,1,1,0,0,1,...,0,1,1,1,0.874096,0.779528,0.394191,0.8,0.642857,0.013759
44511,0.651789,Chainsaw Man,1,0,1,0,1,0,1,0,...,0,1,1,1,0.876990,0.411451,0.212803,0.4,0.857143,0.006054
49926,0.650401,Kimetsu no Yaiba: Mugen Ressha-hen,1,1,1,0,1,0,0,1,...,0,1,1,1,0.861071,0.207605,0.019001,0.8,0.785714,0.003302
41467,0.646189,Bleach: Sennen Kessen-hen,1,1,1,0,1,0,0,1,...,1,1,1,1,0.955137,0.149575,0.093644,0.8,0.857143,0.006604
51019,0.628563,Kimetsu no Yaiba: Katanakaji no Sato-hen,1,1,1,0,1,0,0,1,...,0,1,1,1,0.837916,0.232567,0.040704,0.8,0.928571,0.005504


Most similarities occur on Supernatural and Action genres.

Shounen is the shared demographic. 

Shared rating is R17+

## Compare Similarity to Watched List
Test on watching the Entire JJK Series

In [6]:
df_History = USM[USM['Title'].str.contains(r'Jujutsu Kaisen', case=False, na=False, regex=True)]
df_History.Title

MAL_id
40748                      Jujutsu Kaisen
48561              Jujutsu Kaisen 0 Movie
51009           Jujutsu Kaisen 2nd Season
56243    Jujutsu Kaisen 2nd Season Recaps
Name: Title, dtype: object

#### Similar Features Shared between All Watched Entries

In [7]:
importance = df_History.drop(columns=['Title']).mean(axis=0).values
pd.Series(importance, index=USM.drop(columns=['Title']).columns).sort_values(ascending=False).head(10)

Genre_Action                             1.000000
Theme_School                             1.000000
Producer_TOHO animation                  1.000000
Studio_MAPPA                             1.000000
Is_Finished                              1.000000
Rating_R - 17+ (violence & profanity)    1.000000
Source_Manga                             1.000000
Genre_Supernatural                       1.000000
Demo_Shounen                             1.000000
Score                                    0.861071
dtype: float64

In [8]:
importance = importance.reshape([1,-1])
colMask = np.concatenate([np.array([[True, True]]), importance > 0.25], axis=1).flatten()

In [9]:
ids = pd.Series(cosine_similarity(importance, USM.drop(columns='Title')).flatten(), index=USM.index)
ids = ids.sort_values(ascending=False)
ids = ids.drop(df_History.index)    # Remove already watched

df_Results = USM.loc[ids.keys()].copy()
df_Results.insert(1, 'Similarity', ids.values)
df_Results.loc[:, colMask].head()

,Title,Similarity,Is_Finished,Genre_Supernatural,Genre_Action,Demo_Shounen,Theme_School,Studio_MAPPA,Producer_Shueisha,Producer_Mainichi Broadcasting System,Producer_TOHO animation,Producer_dugout,Producer_Sumzap,Type_TV,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Members,Related_Entries,Year
MAL_id,,,,,,,,,,,,,,,,,,,,
49926,Kimetsu no Yaiba: Mugen Ressha-hen,0.693611,1,1,1,1,0,0,1,0,0,0,0,1,1,1,0.861071,0.207605,0.8,0.785714
44511,Chainsaw Man,0.692087,1,0,1,1,0,1,0,0,0,1,0,1,1,1,0.876990,0.411451,0.4,0.857143
46569,Jigokuraku,0.672287,1,1,1,1,0,1,0,0,0,0,0,1,1,1,0.823444,0.193034,0.4,0.928571
51019,Kimetsu no Yaiba: Katanakaji no Sato-hen,0.671303,1,1,1,1,0,0,1,0,0,0,0,1,1,1,0.837916,0.232567,0.8,0.928571
55701,Kimetsu no Yaiba: Hashira Geiko-hen,0.670792,1,1,1,1,0,0,1,0,0,0,0,1,1,1,0.826339,0.144093,0.8,1.000000


Once again, Demon Slayer and Chainsaw Man are recommended.

# More Tests

In [10]:
class Recommender:
    def __init__(self, isPrintWatched = True, isPrintSimilarity = True):
        self.isPrintWatched = isPrintWatched
        self.isPrintSimilarity = isPrintSimilarity
        
        self.df_History = None
        self.importance = None
        self.df_Results = None
    
    def QueryWatchedDF(self, watchedRegex : str, USM : pd.DataFrame):
        self.df_History = USM[USM['Title'].str.contains(watchedRegex, case=False, na=False, regex=True)]
        
        if self.isPrintWatched:
            print('Watched:')
            print(self.df_History.Title)

    def CalcWatchedSimilarity(self, USM):
        self.importance = self.df_History.drop(columns=['Title']).mean(axis=0).values
        
        if self.isPrintSimilarity:
            print('\nKey Similarities:')
            print(pd.Series(self.importance, index=self.df_History.drop(columns=['Title']).columns).sort_values(ascending=False).head(10))
        
        self.importance = self.importance.reshape([1,-1])
    
    def Recommend(self, watchedRegex : str, USM : pd.DataFrame):
        self.QueryWatchedDF(watchedRegex, USM)
        self.CalcWatchedSimilarity(USM)
    
    
        ids = pd.Series(cosine_similarity(self.importance, USM.drop(columns='Title')).flatten(), index=USM.index)
        ids = ids.sort_values(ascending=False)
        ids = ids.drop(self.df_History.index)    # Remove already watched
        
        self.df_Results = USM.loc[ids.keys()].copy()
        self.df_Results.insert(1, 'Similarity', ids.values)
    
    def GenerateUSMTopFeaturesWeight(self, USM : pd.DataFrame, weightMultiplier = 1):
        if self.importance is None:
            raise ValueError('Recommend first to get important feature values')
        
        top_10_features_indices = self.importance.flatten().argsort()[-10:][::-1] + 1 # Offset by 1 bcuz of 'Title'
        size = top_10_features_indices.size

        USM_weighted = USM.copy()


        prevM = size + 1
        for i in np.arange(0, size):
            curr = self.importance.flatten()[top_10_features_indices[i]]
            prev = self.importance.flatten()[top_10_features_indices[i-1]]
            
            if i > 0 and curr == prev:
                m = prevM
            else:
                m = (prevM - 1) 
                prevM = m

            USM_weighted.iloc[:,top_10_features_indices[i]] = USM.iloc[:,top_10_features_indices[i]] * m * weightMultiplier

            
        
        return USM_weighted
    
    def Show(self, cutoff = 5):
        print(f'\nTop {cutoff} Mean Similarity: ', self.df_Results['Similarity'].head(cutoff).mean())
        
        colMask = np.concatenate([np.array([[True, True]]), self.importance > 0.25], axis=1).flatten()
        return self.df_Results.loc[:, colMask].head(cutoff)

## Modifiying USM
Before proceeding, we will change the USM features. This modification is done because we will be scaling the most important weights, hence having redundant info taking up the top features will be a waste.

1. Remove Is_Finished. It's the most recurring value since most anime are already completed.
2. Remove Producer & Licensor, since most viewers often prioritize only Studio. The only exception, is when the Licensor is Netflix.

In [11]:
def FindColsWithPrefix(arr, prefix):
    indices = set()
    for i, string in enumerate(arr):
        if string.startswith(prefix):
            indices.add(string)
    return indices

cols = {'Is_Finished'} | FindColsWithPrefix(USM.columns, 'Producer') | FindColsWithPrefix(USM.columns, 'Licensor')

USM = USM.drop(columns=list(cols))

## Testing Importance Weights

### Popular Fantasy: Re:Zero

### Ver 1. Original

In [12]:
r = Recommender()

In [13]:
r.Recommend(r'Re:Zero', USM)
r.Show()

Watched:
MAL_id
31240                Re:Zero kara Hajimeru Isekai Seikatsu
33142                     Re:Zero kara Hajimeru Break Time
36286    Re:Zero kara Hajimeru Isekai Seikatsu - Memory...
39921    Re:Zero kara Hajimeru Isekai Seikatsu - Memory...
38414    Re:Zero kara Hajimeru Isekai Seikatsu - Hyouke...
41590    Re:Zero kara Hajimeru Isekai Seikatsu - Hyouke...
42364          Re:Zero kara Hajimeru Break Time 2nd Season
39587     Re:Zero kara Hajimeru Isekai Seikatsu 2nd Season
42203    Re:Zero kara Hajimeru Isekai Seikatsu 2nd Seas...
54857     Re:Zero kara Hajimeru Isekai Seikatsu 3rd Season
60012          Re:Zero kara Hajimeru Break Time 3rd Season
Name: Title, dtype: object

Key Similarities:
Source_Light novel       1.000000
Score                    0.772925
Genre_Fantasy            0.727273
Year                     0.681818
Theme_Isekai             0.545455
Studio_White Fox         0.545455
Related_Entries          0.509091
Studio_Studio PuYUKAI    0.454545
Genre_Suspense   

,Title,Similarity,Genre_Comedy,Genre_Drama,Genre_Fantasy,Genre_Suspense,Theme_Psychological,Theme_Isekai,Theme_Time Travel,Studio_White Fox,Studio_Studio PuYUKAI,Type_Movie,Type_TV,Rating_PG-13 - Teens 13 or older,Rating_R - 17+ (violence & profanity),Source_Light novel,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,,,,,,,
48897,Overlord: Ple Ple Pleiades 4,0.721638,1,0,1,0,0,1,0,0,1,0,0,1,0,1,0.620839,0.4,0.857143
52461,Rougo ni Sonaete Isekai de 8-manmai no Kinka w...,0.714776,0,0,1,0,0,1,0,0,0,0,1,1,0,1,0.657019,0.4,0.928571
33674,No Game No Life: Zero,0.703571,0,1,1,0,0,1,0,0,0,1,0,1,0,1,0.835022,0.8,0.500000
42429,Honzuki no Gekokujou: Shisho ni Naru Tame ni w...,0.690161,0,0,1,0,0,1,0,0,0,0,1,1,0,1,0.820550,0.8,0.857143
45486,Kuma Kuma Kuma Bear Punch!,0.688532,1,0,1,0,0,1,0,0,0,0,1,1,0,1,0.706223,0.8,0.928571


### Ver 2. Scaling the Previously Discovered Most Important Features

In [14]:
r.isPrintWatched = False
r.Recommend(r'Re:Zero', r.GenerateUSMTopFeaturesWeight(USM))
r.Show(10)


Key Similarities:
Source_Light novel                  10.000000
Score                                6.956322
Genre_Fantasy                        5.818182
Year                                 4.772727
Theme_Isekai                         3.272727
Studio_White Fox                     3.272727
Related_Entries                      2.545455
Rating_PG-13 - Teens 13 or older     1.818182
Studio_Studio PuYUKAI                1.363636
Theme_Psychological                  1.363636
dtype: float64

Top 10 Mean Similarity:  0.9424253601726157


,Title,Similarity,Genre_Comedy,Genre_Drama,Genre_Fantasy,Genre_Suspense,Theme_Psychological,Theme_Isekai,Theme_Time Travel,Studio_White Fox,Studio_Studio PuYUKAI,Type_Movie,Type_TV,Rating_PG-13 - Teens 13 or older,Rating_R - 17+ (violence & profanity),Source_Light novel,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,,,,,,,
38659,Shinchou Yuusha: Kono Yuusha ga Ore Tueee Kuse...,0.948896,1,0,8,0,0,6,0,6,0,0,1,0,1,10,6.603473,2.0,4.5
46971,Arifureta Shokugyou de Sekai Saikyou: Prologue,0.946420,0,0,8,0,0,6,0,6,0,0,0,4,0,10,5.574530,2.0,5.0
36882,Arifureta Shokugyou de Sekai Saikyou,0.944279,0,0,8,0,0,6,0,6,0,0,1,4,0,10,5.626628,4.0,4.5
40083,Arifureta Shokugyou de Sekai Saikyou Specials,0.943097,0,0,8,0,0,6,0,6,0,0,0,4,0,10,5.639653,2.0,4.5
42429,Honzuki no Gekokujou: Shisho ni Naru Tame ni w...,0.942471,0,0,8,0,0,6,0,0,0,0,1,4,0,10,7.384949,4.0,6.0
40815,Honzuki no Gekokujou: Shisho ni Naru Tame ni w...,0.941521,0,0,8,0,0,6,0,0,0,0,1,4,0,10,7.397974,4.0,5.0
49458,Kono Subarashii Sekai ni Shukufuku wo! 3,0.940411,1,0,8,0,0,6,0,0,0,0,1,4,0,10,7.762663,4.0,7.0
39468,Honzuki no Gekokujou: Shisho ni Naru Tame ni w...,0.939570,0,0,8,0,0,6,0,0,0,0,1,4,0,10,7.267728,4.0,4.5
38040,Kono Subarashii Sekai ni Shukufuku wo! Movie: ...,0.939097,1,0,8,0,0,6,0,0,0,1,0,4,0,10,7.840810,4.0,4.5


### Ver 3. Try removing the Top 5 Key Features

In [15]:
r = Recommender(False, True)
r.Recommend(r'Re:Zero', USM.drop(columns=['Source_Light novel', 'Related_Entries', 'Score', 'Related_Entries', 'Year']))
r.Show()


Key Similarities:
Genre_Fantasy                       0.727273
Theme_Isekai                        0.545455
Studio_White Fox                    0.545455
Theme_Psychological                 0.454545
Studio_Studio PuYUKAI               0.454545
Genre_Drama                         0.454545
Genre_Suspense                      0.454545
Rating_PG-13 - Teens 13 or older    0.454545
Theme_Time Travel                   0.363636
Type_Movie                          0.363636
dtype: float64

Top 5 Mean Similarity:  0.6283187539620053


,Title,Similarity,Genre_Comedy,Genre_Drama,Genre_Fantasy,Genre_Suspense,Theme_Psychological,Theme_Isekai,Theme_Time Travel,Studio_White Fox,Studio_Studio PuYUKAI,Type_Movie,Type_TV,Rating_PG-13 - Teens 13 or older,Rating_R - 17+ (violence & profanity)
MAL_id,,,,,,,,,,,,,,,
38472,Isekai Quartet,0.633070,1,0,1,0,0,1,0,0,1,0,1,1,0
39988,Isekai Quartet 2,0.632260,1,0,1,0,0,1,0,0,1,0,1,1,0
41567,Isekai Quartet Movie: Another World,0.631330,1,0,1,0,0,1,0,0,1,1,0,1,0
9253,Steins;Gate,0.628740,0,1,0,1,1,0,1,1,0,0,1,1,0
30901,Utawarerumono: Itsuwari no Kamen,0.616194,0,1,1,0,0,0,0,1,0,0,1,1,0


### Profiling Studio Trigger

In [16]:
r = Recommender()
r.Recommend(r'Kill la Kill|Darling in the FranXX|Cyberpunk|Witch Academia', USM)
r.Show()

Watched:
MAL_id
18679                                    Kill la Kill
14349                           Little Witch Academia
21659                           Kill la Kill Specials
19489    Little Witch Academia: Mahoujikake no Parade
33489                      Little Witch Academia (TV)
35849                           Darling in the FranXX
42310                          Cyberpunk: Edgerunners
Name: Title, dtype: object

Key Similarities:
Studio_Trigger                      1.000000
Score                               0.786645
Genre_Fantasy                       0.714286
Theme_School                        0.714286
Genre_Comedy                        0.714286
Genre_Action                        0.571429
Year                                0.428571
Genre_Adventure                     0.428571
Type_TV                             0.428571
Rating_PG-13 - Teens 13 or older    0.428571
dtype: float64

Top 5 Mean Similarity:  0.6857809204185956


,Title,Similarity,Genre_Comedy,Genre_Ecchi,Genre_Action,Genre_Fantasy,Genre_Sci-Fi,Genre_Adventure,Theme_Urban Fantasy,Theme_School,Studio_Trigger,Type_Movie,Type_TV,Rating_G - All Ages,Rating_PG-13 - Teens 13 or older,Score,Year
MAL_id,,,,,,,,,,,,,,,,,
34834,Hina Logi: From Luck & Logic,0.703228,1,0,1,1,0,0,0,1,0,0,1,0,1,0.613603,0.500000
40060,BNA,0.692987,0,0,1,1,0,0,1,0,1,0,1,0,1,0.714906,0.714286
31442,Musaigen no Phantom World,0.678735,1,1,1,1,0,0,1,1,0,0,1,0,1,0.643994,0.428571
37657,Gakuen Basara,0.678379,1,0,1,0,0,0,0,1,0,0,1,0,1,0.551375,0.571429
32681,Uchuu Patrol Luluco,0.675577,1,0,1,0,1,1,0,0,1,0,1,0,1,0.740955,0.428571


### Mixed Bag: Fate Franchise
* Main - Fantasy, action
* Ilya - Magical girls, ecchi
* Grand Carnival - Comedy

In [17]:
r = Recommender()
r.Recommend(r'Fate/', USM)
r.Show()

Watched:
MAL_id
7559                       Fate/stay night TV Reproduction
6922          Fate/stay night Movie: Unlimited Blade Works
10087                                            Fate/Zero
12565                                       Fate/Prototype
11741                                 Fate/Zero 2nd Season
13263             Fate/Zero: Onegai! Einzbern Soudanshitsu
13183                                      Fate/Zero Remix
19109              Fate/kaleid liner Prisma☆Illya Specials
14829                       Fate/kaleid liner Prisma☆Illya
19165                                       Fate/Zero Cafe
22297               Fate/stay night: Unlimited Blade Works
27821      Fate/stay night: Unlimited Blade Works Prologue
25011        Fate/kaleid liner Prisma☆Illya 2wei! Specials
20509                 Fate/kaleid liner Prisma☆Illya 2wei!
18851    Fate/kaleid liner Prisma☆Illya: Undoukai de Da...
31389    Fate/stay night: Unlimited Blade Works 2nd Sea...
31056    Fate/kaleid liner Prisma☆Illya 

,Title,Similarity,Genre_Comedy,Genre_Action,Genre_Fantasy,Theme_Urban Fantasy,Studio_ufotable,Studio_SILVER LINK.,Rating_PG-13 - Teens 13 or older,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,,
45654,Xiyou Ji: Zai Shi Yao Wang,0.763573,0,1,1,0,0,0,1,0,0,0.630970,0.0,0.785714
48585,Black Clover: Mahou Tei no Ken,0.755986,1,1,1,0,0,0,1,0,1,0.808973,0.4,0.928571
38268,Hangyakusei Million Arthur 2nd Season,0.753387,1,1,1,0,0,0,1,0,0,0.562952,0.4,0.642857
37555,Hangyakusei Million Arthur,0.747085,1,1,1,0,0,0,1,0,0,0.522431,0.4,0.571429
50898,Shen Wu Tianzun,0.743045,0,1,1,0,0,0,1,0,1,0.625181,0.0,0.714286


In [18]:
r.isPrintWatched = False
r.Recommend(r'Fate/', r.GenerateUSMTopFeaturesWeight(USM))
r.Show(10)


Key Similarities:
Score                                    6.929915
Genre_Action                             6.061224
Rating_PG-13 - Teens 13 or older         5.387755
Genre_Fantasy                            4.285714
Year                                     2.842566
Genre_Comedy                             2.142857
Source_Manga                             1.469388
Related_Entries                          1.004082
Theme_Urban Fantasy                      0.612245
Rating_R - 17+ (violence & profanity)    0.612245
dtype: float64

 0.9683692281954299ity: 


,Title,Similarity,Genre_Comedy,Genre_Action,Genre_Fantasy,Theme_Urban Fantasy,Studio_ufotable,Studio_SILVER LINK.,Rating_PG-13 - Teens 13 or older,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,,
39551,Tensei shitara Slime Datta Ken 2nd Season,0.973832,5,9,7,0,0,0,8,0,4,8.625181,2.4,4.714286
41487,Tensei shitara Slime Datta Ken 2nd Season Part 2,0.973402,5,9,7,0,0,0,8,0,4,8.552822,2.4,4.714286
48585,Black Clover: Mahou Tei no Ken,0.971840,5,9,7,0,0,0,8,0,4,8.089725,1.2,5.571429
37430,Tensei shitara Slime Datta Ken,0.971528,5,9,7,0,0,0,8,0,4,8.306802,1.2,3.428571
41923,Yi Nian Yong Heng,0.968899,5,9,7,0,0,0,8,0,0,8.017366,1.2,4.285714
40111,Nezha Zhi Mo Tong Jiang Shi,0.966414,5,9,7,0,0,0,8,0,0,7.597685,1.2,3.857143
44406,Da Wang Rao Ming,0.965250,5,9,7,0,0,0,8,0,0,7.424023,1.2,4.714286
43523,Tsuki ga Michibiku Isekai Douchuu,0.964787,5,9,7,0,0,0,8,0,0,7.684515,2.4,4.714286
38790,Itai no wa Iya nanode Bougyoryoku ni Kyokufuri...,0.963875,5,9,7,0,0,1,8,0,0,7.395080,2.4,4.285714


### Fighting + Gore: Baki

In [19]:
r = Recommender()
r.Recommend(r'Baki:|^Baki', USM)
r.Show()

Watched:
MAL_id
33566    Baki: Most Evil Death Row Convicts Special Anime
34443                                                Baki
39555                             Baki: Dai Raitaisai-hen
42940                             Hanma Baki: Son of Ogre
51318                  Hanma Baki: Son of Ogre 2nd Season
Name: Title, dtype: object

Key Similarities:
Theme_Combat Sports                      1.000000
Demo_Shounen                             1.000000
Genre_Sports                             1.000000
Theme_Gore                               1.000000
Rating_R - 17+ (violence & profanity)    1.000000
Source_Manga                             1.000000
Studio_TMS Entertainment                 0.800000
Type_ONA                                 0.800000
Score                                    0.725904
Related_Entries                          0.720000
dtype: float64

 0.6220705327430005ty: 


,Title,Similarity,Genre_Sports,Demo_Shounen,Theme_Gore,Theme_Combat Sports,Studio_TMS Entertainment,Type_ONA,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,
59497,Rising Impact Season 2,0.643563,1,1,0,0,0,1,0,1,0.804631,0.4,1.000000
57487,Rising Impact,0.642153,1,1,0,0,0,1,0,1,0.743849,0.4,1.000000
35789,Yowamushi Pedal: Glory Line,0.610556,1,1,0,0,1,0,0,1,0.749638,0.8,0.571429
31783,Yowamushi Pedal: New Generation,0.607896,1,1,0,0,1,0,0,1,0.756874,0.8,0.500000
54730,Kinnikuman: Kanpeki Chоujin Shiso-hen,0.606186,1,1,0,1,0,0,0,1,0.638205,0.8,1.000000


In [20]:
r.isPrintWatched = False
r.Recommend(r'Baki:|^Baki', r.GenerateUSMTopFeaturesWeight(USM))
r.Show()


Key Similarities:
Theme_Combat Sports                      10.000000
Demo_Shounen                             10.000000
Genre_Sports                             10.000000
Theme_Gore                               10.000000
Rating_R - 17+ (violence & profanity)    10.000000
Source_Manga                             10.000000
Type_ONA                                  7.200000
Studio_TMS Entertainment                  6.400000
Score                                     5.081331
Related_Entries                           4.320000
dtype: float64

Top 5 Mean Similarity:  0.7710060764801


,Title,Similarity,Genre_Sports,Demo_Shounen,Theme_Gore,Theme_Combat Sports,Studio_TMS Entertainment,Type_ONA,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,
54730,Kinnikuman: Kanpeki Chоujin Shiso-hen,0.772476,10,10,0,10,0,0,0,10,4.467438,4.8,1.000000
7793,Ring ni Kakero 1: Kage Dou-hen,0.772307,10,10,0,10,0,0,0,10,4.001447,4.8,0.000000
51009,Jujutsu Kaisen 2nd Season,0.770753,0,10,10,0,0,0,10,10,6.442836,4.8,0.928571
37007,Hinomaruzumou,0.769806,10,10,0,10,0,0,0,10,5.196816,2.4,0.571429
49376,Mou Ippon!,0.769688,10,10,0,10,0,0,0,10,4.751085,2.4,0.928571


In [21]:
r = Recommender()
r.Recommend(r'Jojo', USM)

r.isPrintWatched = False
r.Recommend(r'Jojo', r.GenerateUSMTopFeaturesWeight(USM))
r.Show(10)

Watched:
MAL_id
14719                        JoJo no Kimyou na Bouken (TV)
20899    JoJo no Kimyou na Bouken Part 3: Stardust Crus...
26055    JoJo no Kimyou na Bouken Part 3: Stardust Crus...
31933    JoJo no Kimyou na Bouken Part 4: Diamond wa Ku...
37991       JoJo no Kimyou na Bouken Part 5: Ougon no Kaze
38972    JoJo no Kimyou na Bouken Part 5: Ougon no Kaze...
48661         JoJo no Kimyou na Bouken Part 6: Stone Ocean
53273    JoJo no Kimyou na Bouken Part 6: Stone Ocean P...
51367    JoJo no Kimyou na Bouken Part 6: Stone Ocean P...
Name: Title, dtype: object

Key Similarities:
Demo_Shounen                             1.000000
Genre_Adventure                          1.000000
Studio_David Production                  1.000000
Genre_Action                             1.000000
Rating_R - 17+ (violence & profanity)    1.000000
Source_Manga                             1.000000
Theme_Super Power                        0.888889
Score                                    0.834539
Related

,Title,Similarity,Genre_Action,Genre_Adventure,Demo_Shounen,Theme_Super Power,Studio_David Production,Type_ONA,Type_TV,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,,
39489,Spriggan (ONA),0.930920,10,10,10,0,10,1,0,10,10,5.795948,3.2,0.857143
52741,Undead Unluck,0.922358,10,0,10,10,10,0,7,10,10,6.981187,3.2,1.000000
53998,Bleach: Sennen Kessen-hen - Ketsubetsu-tan,0.869333,10,10,10,0,0,0,7,10,10,8.205499,6.4,0.928571
56784,Bleach: Sennen Kessen-hen - Soukoku-tan,0.869281,10,10,10,0,0,0,7,10,10,8.231548,6.4,1.000000
41467,Bleach: Sennen Kessen-hen,0.869154,10,10,10,0,0,0,7,10,10,8.596237,6.4,0.857143
32370,D.Gray-man Hallow,0.864778,10,10,10,0,0,0,7,10,10,6.876990,3.2,0.428571
46569,Jigokuraku,0.864063,10,10,10,0,0,0,7,10,10,7.410999,3.2,0.928571
37520,Dororo,0.863364,10,10,10,0,0,0,7,10,10,7.619392,3.2,0.642857
5114,Fullmetal Alchemist: Brotherhood,0.863018,10,10,10,0,0,0,7,10,10,8.726483,3.2,0.000000


### Niche Cute Girls Doing Cute Things: Yuru Camp

In [22]:
r = Recommender()
r.Recommend(r'Yuru Camp|Heya Camp', USM)
r.Show()

Watched:
MAL_id
37341                          Yuru Camp△ Specials
34798                                   Yuru Camp△
41061    Heya Camp△: Sauna to Gohan to Sanrin Bike
38476                                   Heya Camp△
49026                 Yuru Camp△ Season 2 Specials
38474                          Yuru Camp△ Season 2
38475                             Yuru Camp△ Movie
53410                          Yuru Camp△ Season 3
58855                 Yuru Camp△ Season 3 Specials
Name: Title, dtype: object

Key Similarities:
Genre_Slice of Life                 1.000000
Theme_CGDCT                         1.000000
Rating_PG-13 - Teens 13 or older    1.000000
Source_Manga                        1.000000
Theme_Iyashikei                     0.888889
Score                               0.788873
Year                                0.777778
Studio_C-Station                    0.777778
Type_Special                        0.444444
Type_TV                             0.444444
dtype: float64

Top 5 Mean Si

,Title,Similarity,Genre_Slice of Life,Theme_CGDCT,Theme_Iyashikei,Studio_C-Station,Type_Special,Type_TV,Rating_PG-13 - Teens 13 or older,Source_Manga,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,
27887,Yama no Susume Second Season Specials,0.857879,1,1,1,0,1,0,1,1,0.646889,0.0,0.285714
48491,Yama no Susume: Next Summit,0.841065,1,1,1,0,0,1,1,1,0.758321,0.4,0.857143
35672,Yama no Susume Third Season,0.823742,1,1,1,0,0,1,1,1,0.752533,0.8,0.571429
45425,Slow Loop,0.812249,1,1,1,0,0,1,1,1,0.701881,0.4,0.857143
39808,Non Non Biyori Nonstop,0.807962,1,1,1,0,0,1,1,1,0.862518,0.8,0.785714


# Summary

The original content-based recommender is able to provide similar anime based on their features. However if we want a more specific target, we could scale the key features to retrieve only the highest matching features.

We also removed noisy and redundant features such as 'Is_Finished', 'Producer_', 'Licensor_'.

# Bag of Words
Another approach is creating features from the synopsis. We extract important words and measure its frequency.

In [23]:
df_BOW = pd.read_csv('Cleaned.csv', index_col=0)[['Title', 'Synopsis']]
df_BOW = df_BOW.loc[~df_BOW.index.duplicated(keep='first')] # Duplicates present from shows continued airing between multiple seasons but still classified as one.
df_BOW = df_BOW.dropna()

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
pd.set_option('display.max_colwidth', None)

matrix_BOW = TfidfVectorizer(stop_words='english').fit_transform(df_BOW['Synopsis'])
cosine_sim = cosine_similarity(matrix_BOW)

def BOW_Recommend(item_index, cosine_sim=cosine_sim):
    similarity_scores = list(enumerate(cosine_sim[item_index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similar_items = [x[0] for x in similarity_scores]
    
    return similar_items


### Test Dandandan
Not very good.

It focuses on the characters names: Momo and Ken.

In [25]:
i = df_BOW.reset_index().query('Title == "Dandadan"').index[0]
recommended_items = BOW_Recommend(i)
df_BOW.iloc[recommended_items[:10]]

,Title,Synopsis
MAL_id,,
57334,Dandadan,"Reeling from her recent breakup, Momo Ayase, a popular high schooler, shows kindness to her socially awkward schoolmate, Ken Takakura, by standing up to his bullies. Ken misunderstands her intentions, believing he has made a new friend who shares his obsession with aliens and UFOs. However, Momo's own eccentric occult beliefs lie in the supernatural realm; she thinks aliens do not exist. A rivalry quickly brews as each becomes determined to prove the other wrong. Despite their initial clash over their opposing beliefs, Momo and Ken form an unexpected but intimate friendship, a bond forged in a series of supernatural battles and bizarre encounters with urban legends and paranormal entities. As both develop unique superhuman abilities, they learn to supplement each other's weaknesses, leading them to wonder if their newfound partnership may be about more than just survival."
30014,Momokuri,"After taking one hundred secret photos and observing him from afar for months, second-year high schooler Yuki Kurihara has finally mustered up the courage to ask out her first-year crush Shinya ""Momo"" Momotsuki. Although taken by surprise, the bashful Momo accepts; however, he does not know the profoundly abnormal truth. As her strait-laced friend, Norika Mizuyama, has observed, Yuki has developed some unnerving—but nonetheless sincere—habits: taking pictures of Momo in secret, doing extensive research into his personal life, collecting his used straws, and even going ""Momo watching."" Though Momo remains blissfully unaware of his new girlfriend's peculiar habits, he does notice some oddities in their daily conversations. Still unsure and nervous about his first relationship, Momo finds himself regularly getting into awkward interactions due to his inexperience, but nevertheless resolves to make his new girlfriend happy. Momokuri follows Yuki and Momo as they shyly explore their newfound love, and also deal with the problems that arise from it."
10389,Momo e no Tegami,"After the unexpected death of her father, 11-year-old Momo Miyaura leaves Tokyo with her mother and moves to an old remote island in Seto Inland Sea. The only memento she has from her father is an unfinished letter with only two words inside: ""Dear Momo""—along with her heart's unrest from it. In the new and unfamiliar small town, Momo reluctantly tries to adjust to the outmoded wooden buildings, silent crop fields, and mysterious isolated shrines. One day, while exploring the attic of her new home, she finds a worn out picture book about youkai. Following this discovery, strange things begin to happen around town, and Momo is greeted by the arrival of three troublesome youkai. Momo e no Tegami tells the story of a young girl as she struggles to adapt to her bizarre new life and ultimately come to terms with her father's mysterious letter."
38561,Pan to Boku no Momo-chan,"The story follows a boy named Shiro and his aunt, Momo, who live together after the death of Shiro's mother, Momo's older sister. Shiro makes bread for Momo in the mornings."
23171,Mahou Shoujo wa Kiss Shite Kawaru,After Iori fell in love with Ken at a young age she wanted to be his wife more than anything. Ken feels the same way about Iori. When Ken is nearly killed by a monster she is told the only way to save him is to have sex with other men to make her more powerful as a magical girl.
51635,Kenda Master Ken,"Kenda Master Ken chronicles the exploits of Ken Tamaki as he battles rivals using kendama, a traditional Japanese skill toy."
51636,Kenda Master Ken (TV),"Ken Tamaki defeats corrupt organizations and even finds love through his favorite pastime, the Japanese skill toy ""kendama."" But kendama is no mere toy. The stakes are high and Ken will have to fight for his right to play in KENDA MASTER KEN!"
39518,Vampire in the Garden,"During a winter long ago, near-immortal vampires began plaguing the world. As their population grew at an astounding rate, they promp

### Test Frieren
It's quite good. It's fixating on `long time ago` followed by `but now settled peaceful`.

In [26]:
i = df_BOW.reset_index().query('Title == "Sousou no Frieren"').index[0]
recommended_items = BOW_Recommend(i)
df_BOW.iloc[recommended_items[:10]]

,Title,Synopsis
MAL_id,,
52991,Sousou no Frieren,"During their decade-long quest to defeat the Demon King, the members of the hero's party—Himmel himself, the priest Heiter, the dwarf warrior Eisen, and the elven mage Frieren—forge bonds through adventures and battles, creating unforgettable precious memories for most of them. However, the time that Frieren spends with her comrades is equivalent to merely a fraction of her life, which has lasted over a thousand years. When the party disbands after their victory, Frieren casually returns to her ""usual"" routine of collecting spells across the continent. Due to her different sense of time, she seemingly holds no strong feelings toward the experiences she went through. As the years pass, Frieren gradually realizes how her days in the hero's party truly impacted her. Witnessing the deaths of two of her former companions, Frieren begins to regret having taken their presence for granted; she vows to better understand humans and create real personal connections. Although the story of that once memorable journey has long ended, a new tale is about to begin."
56885,Sousou no Frieren: ●● no Mahou,"Frieren loves collecting peculiar spells, and her ever-growing arsenal of magic contains the perfect spells for many occasions. Whether it be by stealthily removing alcohol from Heiter's drinks or by remedying her own sleep issues, Frieren is sure to brighten her companions' days with her magic."
38062,Endro~!,"In a world of adventurers and magic lies Naral Island. Every generation, a Demon Lord rises to plague the land, and every generation, a Hero is born to subdue him. For countless centuries, the cycle has repeated with no end in sight. The latest Hero, Juulia ""Yusha"" Charldetto, has almost completed her valiant campaign alongside her party members: responsible priest Seiran ""Seira"" Élénoir, enigmatic mage Meiza ""Mei"" Endust, and hyper-energetic warrior Fai Fai. In the final battle against the Demon Lord, Yusha's party attempt a risky spell to cast their enemy into the drifts of time. But the incantation goes awry, sending Yusha and her friends back to a time before the Demon Lord, before Yusha becomes the Hero, and before the party had even graduated as adventurers. With their memories of the future erased, the four girls restart their ambitions to become the Hero's Party, aspiring to defeat the Demon Lord. However, in a sudden twist of fate, the Demon Lord was also sent back in time with her memories intact. Reduced to the form of a little girl, the Demon Lord takes the name Mao and infiltrates the adventurers' school as a teacher, planning to stop Yusha before she becomes a hero. Thus begins the story of Yusha and her friends, in their quest to defeat the Demon Lord, not knowing that the one they seek is right by their side."
34028,Idol Jihen,"Increasing income divide, creeping environmental pollution, unsolvable waste issues, childcare waiting lists being discussed without those concerned, repeated corruption… The government, smeared by vested interests, can't do a thing against the many problems and sources of discontent. It's in this situation, with Japan cornered with no way out, that idols rise up to save the day! The Heroine Party, Sunlight Party, Starlight Party, Bishoujo Party, Wakaba Party, Subculture New Party, and SOS Party. From these seven idol political parties, the idols who have become National Diet members and representatives for each prefecture will smash through the sense of stagnation covering Japan using the power of song and dance! They'll bring back the smiling faces of the people, and wrap Japan in a glittering aura!!"
18677,Yuusha ni Narenakatta Ore wa Shibushibu Shuushoku wo Ketsui Shimashita.,"Dreaming of becoming a hero and vanquishing the Demon King, Raul Chaser enters the Hero Training Program in pursuit of his ambition. However, when the Demon King is defeated and peace returns to the world, the Hero Training Program is suspended indefinitely, makin